# Barrido MIRD vs ISM — validación de tendencias para diseño de geometría

Extiende la prueba singular a un **factorial** sobre los ejes de diseño, para justificar el uso del simulador ISM en el **diseño de geometría de arreglo** (y su posterior extrapolación a arreglos 2D).

**Ejes barridos** (todos disponibles en MIRD, dist=1 m):
- **RT60**: 160 / 360 / 610 ms
- **Ángulo del target**: 0..90° (voces; interferencia fija a −45°)
- **Fuente target**: varias **voces** (el target sólo puede ser voz)
- **Arreglo**: los 3 spacings medidos `3-3-3-8-3-3-3` (26 cm), `4-4-4-8-4-4-4` (32 cm), `8-8-8-8-8-8-8` (56 cm)

**Métricas** (prioridad): **PESQ**, **STOI**, luego **SI-SDR**.

**Procesadores**: DS, NM-MVDR, Oracle-MVDR. El análisis de tendencias se hace **por procesador por separado**; el **Oracle es el de referencia** porque su métrica no depende de qué micrófono se elija como referencia (mide la capacidad intrínseca de la geometría).

**Idea del argumento:** el diseño de geometría es un problema *relativo* (rankear arreglos, no predecir valores absolutos). Si ISM **preserva las tendencias y el ranking** de spacings/ángulos/RT respecto de MIRD medido, entonces es válido para diseñar geometrías — y el sesgo absoluto (fuente directiva MIRD vs omni pyroomacoustics) es *common-mode* y se cancela en la comparación relativa.

In [ ]:
import os, sys, itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.isdir(os.path.join(PROJECT_ROOT, 'src')):
    PROJECT_ROOT = '/home/matias/Documents/Tesis/Vision-Aided-Beamformer'
SRC = os.path.join(PROJECT_ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(PROJECT_ROOT)

from evaluation.full_benchmark_test_dtln import run_grid_search
from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from propagation.mird_loader import MirdDatasetProvider, generate_mird_linear_array_from_spacing
from evaluation.bf_wrappers import DS, NM_MVDR, ORACLE_MB_MVDR_SOUDEN
print('PROJECT_ROOT =', PROJECT_ROOT, '| imports OK')

## 1. Configuración del barrido

`QUICK=True` corre un subconjunto rápido para una primera pasada. Ponelo en `False` para el factorial completo (¡es largo!). El resultado se cachea en CSV para re-analizar sin recomputar.

In [ ]:
QUICK = True    # True = pasada rápida; False = factorial completo

FS, DURATION = 16000, 8
ISIR_DB = 0
INTERF_CFG = [(-45, 1.0)]                 # interferencia fija a -45deg (distinta del target siempre)
SPACINGS = ['3-3-3-8-3-3-3', '4-4-4-8-4-4-4', '8-8-8-8-8-8-8']
SPACING_APERTURE = {'3-3-3-8-3-3-3': 26, '4-4-4-8-4-4-4': 32, '8-8-8-8-8-8-8': 56}  # cm

VOICES = [  # el target SOLO puede ser voz
    'tools/data/signals/p002_emo_adoration_sentences.wav',
    'tools/data/signals/p008_emo_contentment_sentences.wav',
    'tools/data/signals/p011_emo_anger_sentences.wav',
]
VOICES = [os.path.join(PROJECT_ROOT, v) for v in VOICES]
INTERF = [os.path.join(PROJECT_ROOT, 'tools/data/signals/techno_gated commune.wav')]
DTLN_M1 = os.path.join(PROJECT_ROOT, 'src/dnn_denoise/models/model_quant_1.tflite')
MIRD_ROOT = os.path.join(PROJECT_ROOT, 'tools/data/rirs/mird')

if QUICK:
    RT_LIST   = [0.360, 0.610]
    ANGLES    = [0, 45, 90]
    SOURCES   = VOICES[:1]
else:
    RT_LIST   = [0.160, 0.360, 0.610]
    ANGLES    = [0, 15, 30, 45, 60, 75, 90]
    SOURCES   = VOICES

# param_grid IDÉNTICO para ambos benchmarks (el spacing se barre por fuera)
param_grid = {
    'rt60':            RT_LIST,
    'target_angle':    ANGLES,
    'target_dist':     [1.0],
    'interf_configs':  [INTERF_CFG],
    'source_path':     SOURCES,
    'isir_db':         [ISIR_DB],
    'mismatch_gain':   [0],
    'mismatch_phase':  [0],
    'use_wpe':         [False],
    'error_angle_deg': [0.0],
    'error_distance_m':[0.0],
}
n_cells = len(RT_LIST) * len(ANGLES) * len(SOURCES) * len(SPACINGS)
print(f'Configs por benchmark: {n_cells}  (x2 benchmarks x3 procesadores)')

def make_base_config():
    return {
        'fs': FS, 'duration': DURATION, 't_early': 0.050,
        'array_center': [3.0, 3.0, 1.2], 'snr_db': 60.0,
        'source_path': SOURCES[0], 'interf_paths': INTERF,
        'wpe_taps': 7, 'wpe_delay': 3, 'wpe_alpha': 0.9999,
        'wpe_stft_size': 512, 'wpe_stft_shift': 128,
        'stft_window': 512, 'stft_overlap': 384,
        'eval_references': ['early'],
        'dtln_model_path': DTLN_M1,
    }

def make_processors():
    return {
        'DS': DS(),
        'NM-MVDR': NM_MVDR(min_loading=1e-6, alpha=0.99),
        'Oracle-MVDR': ORACLE_MB_MVDR_SOUDEN(min_loading=1e-6, alpha=0.99, sharpen_exp=1.0),
    }

PROCS_ORDER = ['Oracle-MVDR', 'NM-MVDR', 'DS']   # Oracle primero (headline)
METRICS = ['PESQ', 'STOI', 'SI-SDR']
CACHE_CSV = os.path.join(PROJECT_ROOT, 'tests/ism_validation',
                         f'sweep_long_{"quick" if QUICK else "full"}.csv')

## 2. Correr el barrido (MIRD medido + ISM simulado, por spacing)

Se recorre cada spacing y se corren los dos benchmarks con el mismo `param_grid`. Poné `LOAD_CACHED=True` para saltear el cómputo si el CSV ya existe.

In [ ]:
LOAD_CACHED = False

if LOAD_CACHED and os.path.exists(CACHE_CSV):
    long = pd.read_csv(CACHE_CSV)
    print('Cargado de cache:', CACHE_CSV, long.shape)
else:
    provider = MirdDatasetProvider(root_dir=MIRD_ROOT)
    frames_m, frames_i = [], []
    t0 = time.time()
    for sp in SPACINGS:
        print(f'\n########## SPACING {sp} ##########')
        cfg_m = make_base_config(); cfg_m['mird_spacing'] = sp
        cfg_i = make_base_config(); cfg_i['mird_spacing'] = sp
        cfg_i['geometry_mode'] = 'mird_linear'; cfg_i['room_dims'] = [6.0, 6.0, 2.4]; cfg_i['ray_tracing'] = False
        dm = run_mird_grid_search(param_grid, provider, make_processors(), cfg_m,
                                  os.path.join(PROJECT_ROOT, 'tests/ism_validation/sweep_out_mird'),
                                  None, None, save_catalog=False, apply_dtln_post=False)
        di = run_grid_search(param_grid, None, make_processors(), cfg_i,
                             os.path.join(PROJECT_ROOT, 'tests/ism_validation/sweep_out_ism'),
                             None, None, save_catalog=False, apply_dtln_post=False)
        assert np.allclose(cfg_m['mic_coords'], cfg_i['mic_coords']), f'geom mismatch {sp}'
        dm['spacing'] = sp; di['spacing'] = sp
        frames_m.append(dm); frames_i.append(di)
    DFM = pd.concat(frames_m, ignore_index=True)
    DFI = pd.concat(frames_i, ignore_index=True)
    print(f'\nBarrido completo en {(time.time()-t0)/60:.1f} min')

    # --- Construir tabla larga de comparación ---
    keys = ['processor', 'rt60', 'target_angle', 'source', 'spacing']
    KINDS = ['proc', 'Delta_tot']
    parts = []
    for met in METRICS:
        for kind in KINDS:
            col = f'{kind}_{met}_early'
            if col not in DFM.columns or col not in DFI.columns:
                continue
            mm = DFM[keys + [col]].rename(columns={col: 'MIRD'})
            ii = DFI[keys + [col]].rename(columns={col: 'ISM'})
            mg = mm.merge(ii, on=keys)
            mg['metric'] = met; mg['kind'] = kind
            parts.append(mg)
    long = pd.concat(parts, ignore_index=True)
    long['aperture_cm'] = long['spacing'].map(SPACING_APERTURE)
    long['d'] = long['ISM'] - long['MIRD']
    long.to_csv(CACHE_CSV, index=False)
    print('Guardado:', CACHE_CSV, long.shape)

long.head()

## 3. Fidelidad de tendencia global (por procesador × métrica)

Correlación entre el valor **medido (MIRD)** y el **simulado (ISM)** sobre *todas* las condiciones barridas. `Pearson r` mide acuerdo lineal; `Spearman ρ` mide acuerdo de **ranking** (lo que importa para diseño). `MAE` es el desvío absoluto medio (offset esperado).

In [ ]:
def corr_table(kind='proc'):
    out = []
    for proc in PROCS_ORDER:
        for met in METRICS:
            d = long[(long.processor == proc) & (long.metric == met) & (long.kind == kind)].dropna(subset=['MIRD', 'ISM'])
            if len(d) < 3:
                continue
            out.append({
                'processor': proc, 'metric': met, 'N': len(d),
                'Pearson_r': pearsonr(d.MIRD, d.ISM)[0],
                'Spearman_rho': spearmanr(d.MIRD, d.ISM).correlation,
                'MAE': np.mean(np.abs(d['d'])),
                'bias(ISM-MIRD)': np.mean(d['d']),
            })
    return pd.DataFrame(out)

print('=== Fidelidad de tendencia sobre la SALIDA del procesador (kind=proc) ===')
tbl = corr_table('proc')
display(tbl.round(3))

In [ ]:
# Scatter ISM vs MIRD: filas=procesadores, columnas=métricas; color por spacing
colmap = {sp: c for sp, c in zip(SPACINGS, ['#1b9e77', '#7570b3', '#d95f02'])}
fig, axes = plt.subplots(len(PROCS_ORDER), len(METRICS),
                         figsize=(3.4 * len(METRICS), 3.2 * len(PROCS_ORDER)), squeeze=False)
for r, proc in enumerate(PROCS_ORDER):
    for c, met in enumerate(METRICS):
        ax = axes[r][c]
        d = long[(long.processor == proc) & (long.metric == met) & (long.kind == 'proc')].dropna(subset=['MIRD', 'ISM'])
        for sp in SPACINGS:
            ds = d[d.spacing == sp]
            ax.scatter(ds.MIRD, ds.ISM, s=16, alpha=0.7, color=colmap[sp], label=sp if (r == 0 and c == 0) else None)
        if len(d):
            lo = min(d.MIRD.min(), d.ISM.min()); hi = max(d.MIRD.max(), d.ISM.max())
            ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8)
            rho = spearmanr(d.MIRD, d.ISM).correlation
            ax.set_title(f'{proc} | {met}  (ρ={rho:.2f})', fontsize=9)
        ax.set_xlabel('MIRD (medido)'); ax.set_ylabel('ISM (simulado)')
        ax.grid(ls=':', alpha=0.5)
axes[0][0].legend(fontsize=7, title='spacing')
fig.suptitle('Fidelidad ISM vs MIRD (salida del procesador) — la línea y=x es la réplica perfecta', y=1.005)
fig.tight_layout(); plt.show()

## 4. Tendencia por SPACING (la clave para diseño de geometría)

Promedio de cada métrica sobre (RT × ángulo × fuente) vs apertura del arreglo, MIRD vs ISM. Si las dos curvas **suben/bajan igual** con el spacing, el simulador sirve para elegir geometría.

In [ ]:
for proc in PROCS_ORDER:
    fig, axes = plt.subplots(1, len(METRICS), figsize=(3.6 * len(METRICS), 3.4), squeeze=False)
    for c, met in enumerate(METRICS):
        ax = axes[0][c]
        d = long[(long.processor == proc) & (long.metric == met) & (long.kind == 'proc')]
        g = d.groupby('aperture_cm').agg(MIRD=('MIRD', 'mean'), ISM=('ISM', 'mean'),
                                         MIRD_sd=('MIRD', 'std'), ISM_sd=('ISM', 'std')).reset_index()
        ax.errorbar(g.aperture_cm, g.MIRD, yerr=g.MIRD_sd, marker='o', capsize=3, label='MIRD', color='#2c7fb8')
        ax.errorbar(g.aperture_cm, g.ISM,  yerr=g.ISM_sd,  marker='s', capsize=3, label='ISM',  color='#e6550d')
        ax.set_xticks(list(SPACING_APERTURE.values()))
        ax.set_title(met, fontsize=10); ax.set_xlabel('apertura del arreglo [cm]')
        ax.grid(ls=':', alpha=0.5)
    axes[0][0].legend(fontsize=8); axes[0][0].set_ylabel('métrica (media ± sd)')
    fig.suptitle(f'Tendencia por spacing — {proc}', y=1.02, fontsize=12)
    fig.tight_layout(); plt.show()

## 5. Acuerdo de RANKING de spacings (headline del argumento)

Para cada condición (RT, ángulo, fuente) se rankean los 3 arreglos por métrica y se compara el ranking de ISM contra el de MIRD:
- **Spearman ρ medio**: acuerdo de orden (1 = orden idéntico siempre).
- **% mejor arreglo**: fracción de condiciones donde ISM elige el *mismo* arreglo óptimo que MIRD.

Esto es lo que realmente sostiene "puedo usar ISM para elegir la geometría".

In [ ]:
def ranking_agreement(kind='proc'):
    out = []
    for proc in PROCS_ORDER:
        for met in METRICS:
            d = long[(long.processor == proc) & (long.metric == met) & (long.kind == kind)]
            rhos, best_ok, n = [], 0, 0
            for _, g in d.groupby(['rt60', 'target_angle', 'source']):
                g = g.dropna(subset=['MIRD', 'ISM'])
                if g['spacing'].nunique() < len(SPACINGS):
                    continue
                n += 1
                rho = spearmanr(g['MIRD'], g['ISM']).correlation
                if not np.isnan(rho):
                    rhos.append(rho)
                if g.loc[g.MIRD.idxmax(), 'spacing'] == g.loc[g.ISM.idxmax(), 'spacing']:
                    best_ok += 1
            if n:
                out.append({'processor': proc, 'metric': met, 'N_cond': n,
                            'Spearman_rho_medio': np.mean(rhos) if rhos else np.nan,
                            'pct_mejor_arreglo': 100.0 * best_ok / n})
    return pd.DataFrame(out)

print('=== Acuerdo de ranking de spacings (kind=proc) ===')
display(ranking_agreement('proc').round(2))

## 6. Tendencias por ÁNGULO y por RT (Oracle = headline)

El Oracle es el más representativo (independiente del micrófono de referencia). Verificamos que ISM siga a MIRD también al variar el DOA del target y el RT.

In [ ]:
def trend_by(axis, proc='Oracle-MVDR', kind='proc'):
    fig, axes = plt.subplots(1, len(METRICS), figsize=(3.6 * len(METRICS), 3.4), squeeze=False)
    for c, met in enumerate(METRICS):
        ax = axes[0][c]
        d = long[(long.processor == proc) & (long.metric == met) & (long.kind == kind)]
        g = d.groupby(axis).agg(MIRD=('MIRD', 'mean'), ISM=('ISM', 'mean')).reset_index()
        ax.plot(g[axis], g.MIRD, marker='o', label='MIRD', color='#2c7fb8')
        ax.plot(g[axis], g.ISM,  marker='s', label='ISM',  color='#e6550d')
        ax.set_title(met, fontsize=10); ax.set_xlabel(axis); ax.grid(ls=':', alpha=0.5)
    axes[0][0].legend(fontsize=8); axes[0][0].set_ylabel('métrica (media)')
    fig.suptitle(f'{proc}: tendencia por {axis}', y=1.02, fontsize=12)
    fig.tight_layout(); plt.show()

trend_by('target_angle', 'Oracle-MVDR')
trend_by('rt60',         'Oracle-MVDR')

## 7. Lectura

- **Fidelidad global** (sec. 3): `Spearman ρ` alto por procesador/métrica ⇒ ISM ordena las condiciones como MIRD, aunque haya un `bias` absoluto (esperable por fuente directiva vs omni).
- **Tendencia por spacing** (sec. 4) y **ranking** (sec. 5): si ISM y MIRD mueven la métrica en el mismo sentido con la apertura y coinciden en el *mejor* arreglo, queda justificado usar ISM para **elegir geometría**.
- **Ángulo/RT** (sec. 6): confirma que la fidelidad no depende de un DOA o RT particular.
- El **Oracle** es la evidencia más limpia (no depende del micrófono de referencia); DS/NM-MVDR complementan con procesadores reales.
- **Extrapolación a 2D:** validado el simulador sobre arreglos lineales anclados a MIRD, la extensión a geometrías 2D descansa en que la física del ISM (coherencia inter-micrófono, ya validada en `validate_ism_vs_mird.py`) es la misma; conviene enunciarla como extrapolación fundamentada, no como algo medido.